# Ingestão camada gold

Na camada gold iremos realizar a transformação dos dados da silver em um produto de dados de Market Share

## Fórmula, Granularidade e Premissas

**Fórmula de Market Share:**

```
Market Share (%) = sales_value_brl(produto, semana) / total_sales_value_brl(semana) × 100
```

**Granularidade base:** produto × loja × semana (fato enriquecida)
**Agregações disponíveis:** produto, marca, categoria, loja, rede, território, canal e semana

**Premissas:**
- `sold_volume` representa o volume físico vendido (unidade: unidades)
- `sales_value_brl` é o valor de venda em reais (R$)
- Market Share é calculado sobre o total de `sales_value_brl` por semana
- Dados sem cobertura são identificados via `coverage_status` mas não excluídos do cálculo
- A concatenação dos providers A e B usa `UNION ALL` preservando a origem em `source_table`
- Registros em quarentena são excluídos da silver e portanto não chegam à gold

In [0]:
# Configuração da camada gold
CATALOG = "dataquality_challenge"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
print(f"Schema {GOLD} criado/atualizado")

In [0]:
# Fato enriquecida — JOIN da fato consolidada com dimensões + cobertura (Seção 7: integração fatos e dimensões)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_market_share_enriched AS
SELECT
  f.year_week,
  f.store_id,
  f.ean,
  f.sales_value_brl,
  f.sold_volume,
  f.provider,
  f.source_table,
  l.retailer_name,
  l.channel,
  l.state,
  l.territory_id,
  l.city,
  p.brand,
  p.category,
  p.manufacturer,
  p.product_description,
  c.year,
  c.month,
  c.week,
  cov.expected_stores,
  cov.received_stores,
  CASE
    WHEN cov.file_received = 'N' THEN 'no_coverage'
    WHEN cov.received_stores < cov.expected_stores THEN 'partial_coverage'
    ELSE 'full_coverage'
  END AS coverage_status
FROM {SILVER}.fact_market_share_consolidated f
INNER JOIN {SILVER}.dim_loja l ON f.store_id = l.store_id
INNER JOIN {SILVER}.dim_produto p ON f.ean = p.ean
LEFT JOIN {SILVER}.dim_calendario c ON f.year_week = c.year_week
LEFT JOIN {SILVER}.coverage_provider cov
  ON f.provider = cov.provider
  AND f.year_week = cov.year_week
  AND l.retailer_name = cov.retailer_name
""")
print(f"fact_market_share_enriched: {spark.table(f'{GOLD}.fact_market_share_enriched').count()} registros")

## Market Share por dimensão (GroupBy + Window Function)

Cálculo de Market Share % usando `GROUP BY` + `SUM(...) OVER (PARTITION BY ...)`.

In [0]:
# Market Share por produto × semana (GroupBy + Window Function)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.market_share_by_product AS
SELECT
  year_week, ean, product_description, brand, category,
  SUM(sales_value_brl) AS total_sales_brl,
  SUM(sold_volume) AS total_volume,
  ROUND(
    SUM(sales_value_brl) / SUM(SUM(sales_value_brl)) OVER (PARTITION BY year_week) * 100, 2
  ) AS market_share_pct
FROM {GOLD}.fact_market_share_enriched
GROUP BY year_week, ean, product_description, brand, category
""")
print(f"market_share_by_product: {spark.table(f'{GOLD}.market_share_by_product').count()} registros")

In [0]:
# Market Share por marca × semana
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.market_share_by_brand AS
SELECT
  year_week, brand,
  SUM(sales_value_brl) AS total_sales_brl,
  SUM(sold_volume) AS total_volume,
  ROUND(
    SUM(sales_value_brl) / SUM(SUM(sales_value_brl)) OVER (PARTITION BY year_week) * 100, 2
  ) AS market_share_pct
FROM {GOLD}.fact_market_share_enriched
GROUP BY year_week, brand
""")
print(f"market_share_by_brand: {spark.table(f'{GOLD}.market_share_by_brand').count()} registros")

In [0]:
# Market Share por categoria × semana
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.market_share_by_category AS
SELECT
  year_week, category,
  SUM(sales_value_brl) AS total_sales_brl,
  SUM(sold_volume) AS total_volume,
  ROUND(
    SUM(sales_value_brl) / SUM(SUM(sales_value_brl)) OVER (PARTITION BY year_week) * 100, 2
  ) AS market_share_pct
FROM {GOLD}.fact_market_share_enriched
GROUP BY year_week, category
""")
print(f"market_share_by_category: {spark.table(f'{GOLD}.market_share_by_category').count()} registros")

In [0]:
# Market Share por loja × semana
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.market_share_by_store AS
SELECT
  year_week, store_id, retailer_name, city, state, channel,
  SUM(sales_value_brl) AS total_sales_brl,
  SUM(sold_volume) AS total_volume,
  ROUND(
    SUM(sales_value_brl) / SUM(SUM(sales_value_brl)) OVER (PARTITION BY year_week) * 100, 2
  ) AS market_share_pct
FROM {GOLD}.fact_market_share_enriched
GROUP BY year_week, store_id, retailer_name, city, state, channel
""")
print(f"market_share_by_store: {spark.table(f'{GOLD}.market_share_by_store').count()} registros")

In [0]:
# Market Share por rede (retailer) × semana
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.market_share_by_retailer AS
SELECT
  year_week, retailer_name,
  SUM(sales_value_brl) AS total_sales_brl,
  SUM(sold_volume) AS total_volume,
  ROUND(
    SUM(sales_value_brl) / SUM(SUM(sales_value_brl)) OVER (PARTITION BY year_week) * 100, 2
  ) AS market_share_pct
FROM {GOLD}.fact_market_share_enriched
GROUP BY year_week, retailer_name
""")
print(f"market_share_by_retailer: {spark.table(f'{GOLD}.market_share_by_retailer').count()} registros")

In [0]:
# Market Share por território × semana
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.market_share_by_territory AS
SELECT
  year_week, territory_id,
  SUM(sales_value_brl) AS total_sales_brl,
  SUM(sold_volume) AS total_volume,
  ROUND(
    SUM(sales_value_brl) / SUM(SUM(sales_value_brl)) OVER (PARTITION BY year_week) * 100, 2
  ) AS market_share_pct
FROM {GOLD}.fact_market_share_enriched
GROUP BY year_week, territory_id
""")
print(f"market_share_by_territory: {spark.table(f'{GOLD}.market_share_by_territory').count()} registros")

In [0]:
# Market Share por canal × semana
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.market_share_by_channel AS
SELECT
  year_week, channel,
  SUM(sales_value_brl) AS total_sales_brl,
  SUM(sold_volume) AS total_volume,
  ROUND(
    SUM(sales_value_brl) / SUM(SUM(sales_value_brl)) OVER (PARTITION BY year_week) * 100, 2
  ) AS market_share_pct
FROM {GOLD}.fact_market_share_enriched
GROUP BY year_week, channel
""")
print(f"market_share_by_channel: {spark.table(f'{GOLD}.market_share_by_channel').count()} registros")

## Cobertura e Monitoramento de Data Quality

- **Cobertura:** distingue ausência real de vendas de ausência de cobertura (Seção 8)
- **Monitoramento DQ:** métricas agregadas por classificação e severidade (Seção 7)

In [0]:
# Métricas de cobertura por provider × semana × rede (Seção 8: diferenciar ausência real de vendas de ausência de cobertura)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.coverage_metrics AS
SELECT
  provider,
  year_week,
  retailer_name,
  expected_stores,
  received_stores,
  (expected_stores - received_stores) AS missing_stores,
  ROUND(received_stores / NULLIF(expected_stores, 0) * 100, 1) AS coverage_pct,
  CASE
    WHEN file_received = 'N' THEN 'no_file'
    WHEN received_stores < expected_stores THEN 'partial'
    ELSE 'full'
  END AS coverage_status
FROM {SILVER}.coverage_provider
""")
print(f"coverage_metrics: {spark.table(f'{GOLD}.coverage_metrics').count()} registros")

In [0]:
# Monitoramento DQ — métricas agregadas por classificação e severidade (Seção 7)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dq_monitoring AS
SELECT
  classification,
  severity,
  COUNT(*) AS rule_count,
  SUM(fail_count) AS total_failures,
  ROUND(AVG(percent), 2) AS avg_percent
FROM {SILVER}.dq_results
GROUP BY classification, severity
ORDER BY
  CASE classification
    WHEN 'Corrigido automaticamente' THEN 1
    WHEN 'Quarentena' THEN 2
    WHEN 'Aprovado com alerta' THEN 3
    WHEN 'Aprovado' THEN 4
  END,
  CASE severity
    WHEN 'critical' THEN 1
    WHEN 'high' THEN 2
    WHEN 'medium' THEN 3
    WHEN 'low' THEN 4
  END
""")
print(f"dq_monitoring: {spark.table(f'{GOLD}.dq_monitoring').count()} registros")
display(spark.table(f"{GOLD}.dq_monitoring"))

In [0]:
# Resumo da camada gold
print("=== RESUMO DA CAMADA GOLD ===\n")

gold_tables = [
    "fact_market_share_enriched",
    "market_share_by_product",
    "market_share_by_brand",
    "market_share_by_category",
    "market_share_by_store",
    "market_share_by_retailer",
    "market_share_by_territory",
    "market_share_by_channel",
    "coverage_metrics",
    "dq_monitoring",
]
for t in gold_tables:
    cnt = spark.table(f"{GOLD}.{t}").count()
    print(f"  {t}: {cnt} registros")

# Top 5 produtos por Market Share na última semana
latest_week = spark.table(f"{GOLD}.market_share_by_product").select("year_week").distinct().orderBy("year_week", ascending=False).first()["year_week"]
print(f"\nTop 5 produtos por Market Share — semana {latest_week}:")
display(
    spark.table(f"{GOLD}.market_share_by_product")
    .filter(f"year_week = '{latest_week}'")
    .select("product_description", "brand", "category", "total_sales_brl", "market_share_pct")
    .orderBy("market_share_pct", ascending=False)
    .limit(5)
)

## 5 Insights Relevantes (Seção 10)

Insights sustentados por dados aprovados (camada gold), usando Window Function, GroupBy e agregações temporais.

In [0]:
# Insight 1: Concentração de Market Share — Top 5 marcas na última semana
print("=== INSIGHT 1: Concentração de Market Share por Marca ===")
display(
    spark.table(f"{GOLD}.market_share_by_brand")
    .filter(f"year_week = (SELECT MAX(year_week) FROM {GOLD}.market_share_by_brand)")
    .select("brand", "total_sales_brl", "market_share_pct")
    .orderBy("market_share_pct", ascending=False)
    .limit(5)
)

# Insight 2: Evolução temporal do Market Share por categoria (top 3 categorias por semana)
print("\n=== INSIGHT 2: Evolução do Market Share por Categoria ===")
display(
    spark.sql(f"""
    WITH ranked AS (
      SELECT year_week, category, SUM(market_share_pct) AS category_share,
        ROW_NUMBER() OVER (PARTITION BY year_week ORDER BY SUM(market_share_pct) DESC) AS rnk
      FROM {GOLD}.market_share_by_category
      GROUP BY year_week, category
    )
    SELECT year_week, category, ROUND(category_share, 2) AS category_share
    FROM ranked
    WHERE rnk <= 3
    ORDER BY year_week, category_share DESC
    """)
)

# Insight 3: Comparação de cobertura entre Providers A e B
print("\n=== INSIGHT 3: Cobertura por Provider ===")
display(
    spark.sql(f"""
    SELECT provider,
      ROUND(AVG(coverage_pct), 1) AS avg_coverage_pct,
      SUM(CASE WHEN coverage_status = 'full' THEN 1 ELSE 0 END) AS full_weeks,
      SUM(CASE WHEN coverage_status = 'partial' THEN 1 ELSE 0 END) AS partial_weeks,
      SUM(CASE WHEN coverage_status = 'no_file' THEN 1 ELSE 0 END) AS no_file_weeks
    FROM {GOLD}.coverage_metrics
    GROUP BY provider
    ORDER BY provider
    """)
)

# Insight 4: Correlação entre cobertura e volume de vendas por semana
print("\n=== INSIGHT 4: Cobertura vs Volume de Vendas ===")
display(
    spark.sql(f"""
    WITH cov AS (
      SELECT year_week, provider, ROUND(AVG(coverage_pct), 1) AS avg_coverage_pct
      FROM {GOLD}.coverage_metrics
      GROUP BY year_week, provider
    ),
    sales AS (
      SELECT year_week, provider, ROUND(SUM(sales_value_brl), 0) AS total_sales_brl
      FROM {GOLD}.fact_market_share_enriched
      GROUP BY year_week, provider
    )
    SELECT c.year_week, c.provider, c.avg_coverage_pct, s.total_sales_brl
    FROM cov c
    JOIN sales s ON c.year_week = s.year_week AND c.provider = s.provider
    ORDER BY c.year_week, c.provider
    """)
)

# Insight 5: Market Share por canal — dominância na última semana
print("\n=== INSIGHT 5: Market Share por Canal ===")
display(
    spark.table(f"{GOLD}.market_share_by_channel")
    .filter(f"year_week = (SELECT MAX(year_week) FROM {GOLD}.market_share_by_channel)")
    .select("channel", "total_sales_brl", "market_share_pct")
    .orderBy("market_share_pct", ascending=False)
)